In [1]:
import pandas as pd
from src_RF_DT import *
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1.0 - Classificação de Presença Baseado em Fatores Socioeconômicos Usando Random Forest

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [3]:
df = pre_processor_rf_dt(df, objetivo = '', n_samples = 150_000)

## 1.2 - Construção da Matriz X e Vetor y

In [4]:
X = df.drop(['FALTOU'], axis=1)

y = df['FALTOU']

## 1.3 - Separação em Dados de Treino e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.4 -Treinando o modelo

In [6]:
clf = RandomForestClassifier()

clf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, clf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  clf.predict(X_val))))

print(classification_report(y_val, clf.predict(X_val)))

Ein:  0.0182
Eval: 0.3361
              precision    recall  f1-score   support

           0       0.71      0.79      0.75     14895
           1       0.57      0.46      0.51      8996

    accuracy                           0.66     23891
   macro avg       0.64      0.62      0.63     23891
weighted avg       0.65      0.66      0.66     23891



## 1.5 Treinando com os Melhores Parâmetros

In [7]:
cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_features='log2',
                       min_samples_leaf=10, min_samples_split=20,
                       n_estimators=166, random_state=42)
Ein:  0.2901
Eout: 0.3367
              precision    recall  f1-score   support

           0       0.77      0.66      0.71     14895
           1       0.54      0.67      0.60      8996

    accuracy                           0.66     23891
   macro avg       0.66      0.67      0.65     23891
weighted avg       0.68      0.66      0.67     23891



## 1.6 - Treinando com todos os dados

In [9]:
rf = RandomForestClassifier(
    **cv_rf.best_params_,
    class_weight='balanced',  
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

#joblib.dump(rf, 'rf_presenca.pkl')

Ein:  0.2897
Eout: 0.3332
              precision    recall  f1-score   support

           0       0.77      0.67      0.72     18710
           1       0.54      0.66      0.60     11154

    accuracy                           0.67     29864
   macro avg       0.66      0.67      0.66     29864
weighted avg       0.69      0.67      0.67     29864



In [10]:
def tune_random_forest(x_train, y_train, n_iter=10, cv=5, scoring='f1_weighted', random_state=42):
    
    n_estimators = [int(x) for x in np.linspace(start=50, stop=200, num=10)]
    max_features = ['sqrt', 'log2']
    max_depth = [int(x) for x in np.linspace(start=10, stop=40, num=4)]
    max_depth.append(None)
    
    param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_features':      ['sqrt', 'log2', 0.3],  # adiciona fração
    'max_depth':         [5, 10, 15, 20],         # valores menores
    'min_samples_split': [20, 50, 100],            # mais restritivo
    'min_samples_leaf':  [20, 50, 100],            # mais restritivo
    'max_samples':       [0.6, 0.8, 1.0],         # subsampling de linhas
}

    rf = RandomForestClassifier(class_weight='balanced', random_state=random_state)
    
    cv_rf = RandomizedSearchCV(      # ← mudou
        estimator=rf,
        param_distributions=param_grid,  # ← mudou
        n_iter=n_iter,               # ← agora usa o parâmetro
        cv=cv,
        scoring=scoring,
        verbose=2,
        n_jobs=-1,
        random_state=random_state    # ← reprodutibilidade
    )

    cv_rf.fit(x_train, y_train)

    return cv_rf


cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_depth=10, max_features=0.3,
                       max_samples=0.6, min_samples_leaf=20,
                       min_samples_split=100, random_state=42)
Ein:  0.3323
Eout: 0.3330
              precision    recall  f1-score   support

           0       0.78      0.65      0.71     14895
           1       0.55      0.70      0.61      8996

    accuracy                           0.67     23891
   macro avg       0.66      0.67      0.66     23891
weighted avg       0.69      0.67      0.67     23891



In [11]:
joblib.dump(rf, 'rf_presenca_teste.pkl')

['rf_presenca_teste.pkl']